# BM25 Retriever Tests
Small, deterministic checks for BM25 behavior on a toy corpus.

In [1]:
import os
import sys
import pandas as pd
import nltk

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

pd.set_option('display.max_rows', 1000)
pd.set_option("max_colwidth", 40)

# data_dir = os.path.join(project_dir, "data")
# utils_dir = os.path.join(project_dir, "source", "utils")

# for path in (project_dir, utils_dir):
#     if path not in sys.path:
#         sys.path.append(path)
os.chdir(r'C:\Users\ADMIN\Documents\Nam_3\CS419\project')

from source.utils.textPreprocessing import TextPreprocessing
from source.utils.inverted_index import InvertedIndex
from source.utils.BM25_retriever import BM25Retriever

documents_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\cranfield\documents.csv'
queries_path = r'C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\cranfield\queries.csv'

documents = pd.read_csv(documents_path)
queries = pd.read_csv(queries_path)

tp = TextPreprocessing()
processed = tp.process_dataframe(documents)
vocab = tp.get_vocab()

index = InvertedIndex(processed, vocab)
index._build()

bm25 = BM25Retriever(processed, index)

In [2]:
# Basic checks on real dataset
assert len(documents) > 0
assert len(queries) > 0

# Run a few sample queries
sample_queries = queries.head(5).to_dict(orient="records")
results = []
for row in sample_queries:
    qid = row["id"]
    query_text = row["content"]
    hits = bm25.search(query_text, top_k=30)
    results.append({"qid": qid, "query": query_text, "top5": hits})

results_df = pd.DataFrame(results)
results_df

,qid,query,top5
0,1,what similarity laws must be obeyed...,"[(486, 19.58115926640203), (184, 19...."
1,2,what are the structural and aeroela...,"[(12, 28.473374692881826), (51, 17.1..."
2,3,what problems of heat conduction in...,"[(485, 25.601391974150093), (5, 23.9..."
3,4,can a criterion be developed to sho...,"[(488, 28.43412484048587), (166, 27...."
4,5,what chemical kinetic system is app...,"[(103, 15.986925419436375), (1032, 1..."


# Evaluation Metrics (REL Ground Truth)
Compute MAP@10, NDCG@10, Precision/Recall/F1 at 10 using Cranfield REL files.

In [3]:
import os

from source.utils.evaluations import Evaluator
project_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))

rel_dir = r"C:\Users\ADMIN\Documents\Nam_3\CS419\project\data\cranfield\REL"


def load_relations(rel_folder):
    qrels = {}
    for name in os.listdir(rel_folder):
        if not name.endswith(".txt"):
            continue
        path = os.path.join(rel_folder, name)
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) < 3:
                    continue
                qid = int(parts[0])
                doc_id = int(parts[1])
                rel = float(parts[2])
                qrels.setdefault(qid, {})[doc_id] = rel
    return qrels


qrels = load_relations(rel_dir)



In [4]:
k = 25
queries_records = queries.to_dict(orient="records")

bm25_evaluator = Evaluator(
    queries=queries_records,
    qrels=qrels,
    retriever=bm25,
    normalize_relevance_scores=True,
)

result = bm25_evaluator.evaluate_all(top_k=k, return_results=True, return_df=True)
metrics = result.summary

metrics

{'Precision@25': 0.1368888888888889,
 'Recall@25': 0.5372652072729471,
 'F1@25': 0.20329746513293717,
 'MAP@25': 0.2676822541843407,
 'Ndcg@25': 0.35163191159531254}

In [5]:
per_query_df = result.per_query_df
display(per_query_df)

,precision,recall,f1,ap,ndcg,hit,rr,relevant_count,retrieved_count,hit_count,has_relevance
query_id,,,,,,,,,,,
1,0.28,0.250000,0.264151,0.155333,0.259700,1.0,0.500000,28.0,25.0,7.0,1.0
2,0.20,0.208333,0.204082,0.154688,0.159765,1.0,1.000000,24.0,25.0,5.0,1.0
3,0.32,1.000000,0.484848,0.683070,0.796345,1.0,0.500000,8.0,25.0,8.0,1.0
4,0.08,1.000000,0.148148,0.450000,0.624051,1.0,0.500000,2.0,25.0,2.0,1.0
5,0.12,0.750000,0.206897,0.178046,0.243280,1.0,0.250000,4.0,25.0,3.0,1.0
6,0.04,0.250000,0.068966,0.125000,0.272480,1.0,0.500000,4.0,25.0,1.0,1.0
7,0.08,0.400000,0.133333,0.130000,0.211234,1.0,0.250000,5.0,25.0,2.0,1.0
8,0.04,0.090909,0.055556,0.090909,0.040567,1.0,1.000000,11.0,25.0,1.0,1.0
9,0.12,1.000000,0.214286,0.755556,0.885460,1.0,1.000000,3.0,25.0,3.0,1.0
